# Семинар 11. Как классы работают вместе

В десятой лекции мы писали отдельные классы. Теперь один объект будет поручать часть работы другому. Посчитаем доставку, сменим правило расчёта и отправим готовый текст получателю. В каждом случае проверим, что клиент пользуется только обещанными методами и что неудачная операция не портит уже накопленные данные.

## Цели

- передавать объекту готовую зависимость вместо создания её внутри метода;
- подставлять разные реализации одного небольшого договора;
- проверять совместную работу классов с помощью простых объектов-замен;
- выбирать только те методы, которые действительно нужны клиенту;
- проверять ограничения до изменения состояния;
- различать наследование и сотрудничество отдельных объектов.

## Перед началом

Нужны классы, свойства, исключения и [лекция 11](lecture.ipynb). Все задания работают на Python 3.14 со стандартной библиотекой, без файлов и интернета. Собственные `ABC` и `Protocol` не требуются: достаточно объектов с описанными методами.

В заготовках стоит `pass`. Сначала допишите решение, затем выполните ячейку с проверкой и снимите комментарий с `check_task_N()`. Исходный `Run All` проходит, потому что проверки ещё не вызваны.

Типы аргументов соответствуют описанию. Все суммы — целые копейки, вес — целые граммы. Проверяйте только перечисленные ошибки содержимого. Проверяется тип исключения, а не текст. Методы ничего не печатают; если другой возврат не указан, они возвращают `None`.

Объекты-зависимости уже умеют выполнять свою работу. Не переписывайте их алгоритмы внутри клиента и не выбирайте реализацию по имени класса. Клиент должен обращаться к оговорённому методу. Свойства только для чтения не имеют setter; внутренние атрибуты с `_` намеренно не трогаем.

In [ ]:
def expect_error(error_type, action):
    try:
        action()
    except error_type:
        return
    raise AssertionError(f"Ожидалось исключение {error_type.__name__}")

## Задание 1. Два способа посчитать доставку

Одна служба берёт фиксированную сумму, другая — базовую плату и доплату за каждый начатый килограмм. Напишите два независимых класса:

- `FixedDelivery(cost)` хранит неотрицательную цену. `price(weight_grams)` возвращает её.
- `PerKilogramDelivery(base, per_kg)` хранит неотрицательную базовую плату и цену за начатый килограмм. `price(weight_grams)` возвращает `base + число_начатых_килограммов * per_kg`.

У обеих реализаций одинаковое правило для входа: вес должен быть строго положительным, иначе нужен `ValueError`. Отрицательная настройка конструктора тоже означает `ValueError`. Ноль в настройках допустим. Например, 1000 граммов — один килограмм, 1001 грамм — уже два.

Числа по модулю до 10⁹, до 100 000 вызовов. Создание и каждый расчёт — `O(1)` времени и памяти. Общий родительский класс не нужен.

In [ ]:
class FixedDelivery:
    def __init__(self, cost: int) -> None:
        pass

    def price(self, weight_grams: int) -> int:
        pass


class PerKilogramDelivery:
    def __init__(self, base: int, per_kg: int) -> None:
        pass

    def price(self, weight_grams: int) -> int:
        pass

In [ ]:
def check_task_1():
    fixed = FixedDelivery(150)
    per_kg = PerKilogramDelivery(100, 25)
    assert [fixed.price(w) for w in [1, 999, 1000, 1001]] == [150] * 4
    assert [per_kg.price(w) for w in [1, 999, 1000, 1001]] == [125, 125, 125, 150]
    assert FixedDelivery(0).price(1) == 0
    assert PerKilogramDelivery(0, 0).price(1) == 0
    for delivery in [fixed, per_kg]:
        for bad in [0, -1]:
            expect_error(ValueError, lambda: delivery.price(bad))
        assert delivery.price(1000) in [150, 125]
    for args in [(-1, 0), (0, -1)]:
        expect_error(ValueError, lambda: PerKilogramDelivery(*args))
    expect_error(ValueError, lambda: FixedDelivery(-1))
    print("Задание 1: проверки пройдены")

# check_task_1()

## Задание 2. Касса получает готовое правило

Напишите `Checkout(delivery)`. Переданный объект имеет метод `price(weight_grams) -> int`: для положительного веса он возвращает неотрицательную стоимость доставки. Он также может поднять исключение.

Метод `total(subtotal, weight_grams)` возвращает сумму стоимости товаров и доставки. Перед обращением к зависимости проверьте: `subtotal` неотрицателен, вес строго положителен. При нарушении нужен `ValueError`, а метод зависимости не вызывается.

Для допустимого входа вызовите `delivery.price` ровно один раз с исходным весом. Любое исключение из зависимости передаётся вызывающему коду. Сохраните именно переданный объект: если его настройки изменят снаружи, следующий расчёт увидит новое поведение. Тип класса проверять не нужно.

Можно использовать обе реализации из задания 1. В проверках будет и третья, о которой `Checkout` ничего не знает. Числа по модулю до 10⁹; до 100 000 вызовов. Собственная работа `Checkout` — `O(1)` на вызов, без учёта работы зависимости.

In [ ]:
class Checkout:
    def __init__(self, delivery) -> None:
        pass

    def total(self, subtotal: int, weight_grams: int) -> int:
        pass

In [ ]:
def check_task_2():
    class RecordingDelivery:
        def __init__(self):
            self.calls = []
            self.cost = 7
        def price(self, weight_grams):
            self.calls.append(weight_grams)
            return self.cost

    delivery = RecordingDelivery()
    checkout = Checkout(delivery)
    assert checkout.total(100, 1200) == 107
    assert delivery.calls == [1200]
    delivery.cost = 0
    assert checkout.total(0, 1) == 0
    assert delivery.calls == [1200, 1]
    for subtotal, weight in [(-1, 1), (0, 0), (0, -1), (-1, 0)]:
        expect_error(ValueError, lambda: checkout.total(subtotal, weight))
    assert delivery.calls == [1200, 1]

    problem = RuntimeError("Сервис недоступен")
    class UnavailableDelivery:
        def price(self, weight_grams):
            raise problem
    try:
        Checkout(UnavailableDelivery()).total(10, 1)
    except RuntimeError as error:
        assert error is problem
    else:
        raise AssertionError("Ошибка зависимости должна выйти наружу")
    print("Задание 2: проверки пройдены")

# check_task_2()

## Задание 3. Квадрат без чужих обещаний

У прямоугольника из десятой лекции `resize(width, height)` допускает любые два положительных размера. У квадрата другая операция: изменить одну сторону. Сделаем квадрат отдельным классом `Square(side)`, без наследования от прямоугольника.

- Сторона — строго положительное целое. Неположительное значение при создании или в `resize(side)` означает `ValueError`.
- Свойство `side` доступно только для чтения.
- `area()` возвращает площадь.
- `resize(side)` меняет сторону и возвращает `None`. Неудача сохраняет прежнее значение.

Напишите также `total_area(figures)`. Она получает список объектов с методом `area() -> int`, вызывает его один раз у каждого объекта и возвращает сумму. Площади гарантированно неотрицательные целые. Пустой список даёт 0. Другие методы и поля фигур клиенту не нужны, сами фигуры он не меняет.

Такой клиент принимает и квадрат, и прямоугольник: ему обещали только площадь. Обещания общего `resize(width, height)` здесь нет.

До 100 000 фигур, размеры до 10⁹. Операции квадрата — `O(1)`, `total_area` — `O(n)` времени и `O(1)` дополнительной памяти без учёта работы `area`.

In [ ]:
class Square:
    def __init__(self, side: int) -> None:
        pass

    @property
    def side(self) -> int:
        pass

    def area(self) -> int:
        pass

    def resize(self, side: int) -> None:
        pass


def total_area(figures: list) -> int:
    pass

In [ ]:
def check_task_3():
    square = Square(3)
    other = Square(2)
    assert square.side == 3 and square.area() == 9
    assert square.resize(5) is None
    assert square.area() == 25 and other.area() == 4
    for bad in [0, -1]:
        expect_error(ValueError, lambda: Square(bad))
        expect_error(ValueError, lambda: square.resize(bad))
        assert square.side == 5
    expect_error(AttributeError, lambda: setattr(square, "side", 10))

    class RectangleForCheck:
        def __init__(self):
            self.calls = 0
        def area(self):
            self.calls += 1
            return 12
    rectangle = RectangleForCheck()
    assert total_area([square, rectangle, other]) == 41
    assert rectangle.calls == 1
    assert total_area([]) == 0
    assert square.side == 5 and other.side == 2
    print("Задание 3: проверки пройдены")

# check_task_3()

## Задание 4. Подготовить текст и передать получателю

Напишите `ReceiptService(formatter, outbox)` с методом `send(total) -> str`.

У зависимостей маленькие договоры:

- `formatter.format(total) -> str` готовит текст по неотрицательной сумме;
- `outbox.write(text) -> None` принимает готовую строку.

Сначала проверьте `total`: отрицательное значение означает `ValueError` без вызова зависимостей. Затем один раз получите строку у форматтера и один раз передайте **эту же строку** получателю. Верните её вызывающему коду. Своего форматирования внутри сервиса нет.

Если форматтер поднимет исключение, получатель не вызывается. Ошибки обеих зависимостей передаются дальше. Получателю могут быть нужны только записи в список в памяти; метод `close` и работа с файлами здесь не требуются.

Числа по модулю до 10⁹, до 100 000 вызовов. Собственная работа сервиса — `O(1)` без учёта зависимостей.

In [ ]:
class ReceiptService:
    def __init__(self, formatter, outbox) -> None:
        pass

    def send(self, total: int) -> str:
        pass

In [ ]:
def check_task_4():
    events = []
    class Formatter:
        def format(self, total):
            events.append(("format", total))
            return f"Итого {total} коп."
    class Outbox:
        def write(self, text):
            events.append(("write", text))
    service = ReceiptService(Formatter(), Outbox())
    assert service.send(0) == "Итого 0 коп."
    assert events == [("format", 0), ("write", "Итого 0 коп.")]
    expect_error(ValueError, lambda: service.send(-1))
    assert len(events) == 2

    problem = LookupError("Нет шаблона")
    class BrokenFormatter:
        def format(self, total):
            raise problem
    try:
        ReceiptService(BrokenFormatter(), Outbox()).send(10)
    except LookupError as error:
        assert error is problem
    else:
        raise AssertionError("Ошибка форматтера должна сохраниться")
    assert len(events) == 2

    class BrokenOutbox:
        def write(self, text):
            raise problem
    try:
        ReceiptService(Formatter(), BrokenOutbox()).send(10)
    except LookupError as error:
        assert error is problem
    else:
        raise AssertionError("Ошибка получателя должна сохраниться")
    assert events[-1] == ("format", 10)
    print("Задание 4: проверки пройдены")

# check_task_4()

## Задание 5. В журнал попадают только рассчитанные заказы

Напишите `OrderRegister(pricer)`. У переданного объекта есть метод `total(subtotal, weight_grams) -> int`, как у `Checkout` из задания 2. При допустимых аргументах он возвращает неотрицательную итоговую сумму или поднимает исключение.

Метод `add(number, subtotal, weight_grams) -> int`:

- убирает пробелы по краям номера; пустой номер, повторный номер, отрицательная сумма товаров или неположительный вес означают `ValueError` **до вызова** зависимости;
- вызывает расчёт ровно один раз с исходной суммой и весом;
- после успешного расчёта сохраняет пару `(очищенный_номер, итог)` и возвращает итог.

При ошибке расчёта новая запись не появляется и номер остаётся свободным. `snapshot() -> list[tuple[str, int]]` возвращает новый список записей в порядке успешных добавлений. Входные строки неизменяемы, а изменение выданного списка не меняет журнал. У разных регистров свои записи.

До 100 000 добавлений, длина номера до 100 символов, числа по модулю до 10⁹. Проверка повторного номера и собственная работа успешного `add` — ожидаемая амортизированная `O(1)` без учёта расчёта; `snapshot` — `O(k)` по числу записей. Память — `O(k)`.

In [ ]:
class OrderRegister:
    def __init__(self, pricer) -> None:
        pass

    def add(self, number: str, subtotal: int, weight_grams: int) -> int:
        pass

    def snapshot(self) -> list[tuple[str, int]]:
        pass

In [ ]:
def check_task_5():
    class Pricer:
        def __init__(self):
            self.calls = []
            self.fail = False
        def total(self, subtotal, weight_grams):
            self.calls.append((subtotal, weight_grams))
            if self.fail:
                raise ValueError("Не удалось рассчитать доставку")
            return subtotal + 7
    pricer = Pricer()
    register = OrderRegister(pricer)
    other = OrderRegister(pricer)
    assert register.add(" A ", 0, 1) == 7
    assert register.snapshot() == [("A", 7)]
    for args in [("A", 1, 1), (" A ", 1, 1), (" ", 1, 1), ("B", -1, 1), ("B", 0, 0)]:
        expect_error(ValueError, lambda: register.add(*args))
    assert pricer.calls == [(0, 1)]
    pricer.fail = True
    expect_error(ValueError, lambda: register.add("B", 10, 2))
    assert register.snapshot() == [("A", 7)]
    pricer.fail = False
    assert register.add("B", 10, 2) == 17
    view = register.snapshot()
    view.clear()
    assert register.snapshot() == [("A", 7), ("B", 17)]
    assert other.snapshot() == []
    assert register.snapshot() is not register.snapshot()
    print("Задание 5: проверки пройдены")

# check_task_5()

## Итоги

Мы меняли правила расчёта, сохраняя способ обращения к ним. Клиентам хватало маленьких договоров: посчитать цену, узнать площадь, подготовить строку или принять её. Для проверки не понадобились сеть и настоящие файлы — небольшие объекты показали, какой метод был вызван и с какими аргументами.

Обсудите: где у каждого класса повод для изменения? Какая зависимость заменяется без переписывания клиента? Какой метод пришлось бы добавить в слишком широкий интерфейс и почему нашему клиенту он не нужен? Что успело измениться перед ошибкой?

Дальше — [задачи для самостоятельного решения](tasks.md).